# JARVISEO 손 박스 검출기 학습 — h1 (#256 ⑤-b)

**쓰는 법**: 런타임 → 런타임 유형 변경 → **A100 GPU**(없으면 L4) → 저장 → 런타임 → **모두 실행**.
고용량 RAM 토글이 보이면 켜고, 없으면 그대로 둔다(셀 1 이 CPU 수를 찍고 보고서에 남긴다).
중간에 Google Drive 연결 허락 창이 한 번 뜬다. 그 밖에 고를 것은 없다.

- 입력: 내 드라이브 `JARVISEO/hand_box/JARVISEO_HAND_BOX_H1_data.zip` — 맥에서 `tools/hand_box_labels.py build` 로 만든 학습 묶음
  - `yolo_h1`: 손 영상 · 검출기 영상 프레임 · s05 의 자동 정답(MediaPipe + 조각 + 이어 읽기, 사람 검수 없음). 검증 = 손 영상 loby · 검출기 d05
  - `yolo_h1/prep_report.txt`: 묶음을 만들 때 찍힌 기록(장수 · 박스 수)
- 결과(Drive `JARVISEO/hand_box/`): `runs/h1`(가중치 · 그래프) · **`HAND_BOX_H1_report.txt`(이 실행의 기록 전부)**
- 결과는 덮어쓰지 않는다. 이미 있으면 셀 2 에서 멈춘다.
- 레포는 필요 없다(ultralytics 만 설치한다).

In [ ]:
# 1. 설정 · 도움 함수 · GPU 확인
import csv
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from pathlib import Path

ROUND = "h1"
DRIVE = Path("/content/drive/MyDrive/JARVISEO/hand_box")
ZIP = DRIVE / f"JARVISEO_HAND_BOX_{ROUND.upper()}_data.zip"
WORK = Path("/content/work")  # 압축은 Colab 로컬 디스크에 푼다(Drive 보다 읽기가 빠르다)
DATA = WORK / f"yolo_{ROUND}"
RUN_DIR = DRIVE / "runs" / ROUND
REPORT = DRIVE / f"HAND_BOX_{ROUND.upper()}_report.txt"
ULTRALYTICS = "8.4.155"  # 맥에서 손 측정에 쓰는 버전과 같게 고정한다
# 학습 사진이 검출기 R3 의 약 5배라 epoch 를 반으로 줄여도 학습량은 2.5배다.
EPOCHS = 50


def note(text=""):
    """화면에 찍고 보고서 파일에도 남긴다."""
    print(text)
    with REPORT.open("a", encoding="utf-8") as f:
        f.write(text + "\n")


if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "GPU 가 없습니다. 런타임 유형을 A100(없으면 L4)으로 바꾸고 다시 모두 실행하세요."
    )
GPU = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()
RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
CPUS = os.cpu_count()
# 처음 학습하는 모델이라 맞출 이전 회차가 없다. GPU 종류 · CPU 수로 멈추지 않고 보고서에 남긴다.
# 데이터 준비 워커 = min(CPU, 8) 이라 CPU 가 적으면 느리다(T4 가 CPU 2개로 뜬 적이 있다, 11/2).
# 다음 회차는 보고서의 GPU · CPU 와 같게 맞춘다 — CPU 수가 바뀌면 같은 seed 여도 결과가 달라진다.
print(f"GPU: {GPU} · CPU {CPUS}개 · RAM {RAM_GB:.0f}GB")
if CPUS < 8:
    print(f"주의: CPU {CPUS}개 — 데이터 준비가 느려 학습이 오래 걸린다(A100 · L4 는 보통 8개+).")

In [ ]:
# 2. Drive 연결 · 덮어쓰기 막기 (허락 창이 뜨면 내 계정으로 허용)
from google.colab import drive

drive.mount("/content/drive")
if not ZIP.exists():
    raise RuntimeError(
        f"학습 묶음이 없습니다: {ZIP} — 내 드라이브 JARVISEO/hand_box 에 zip 을 올립니다."
    )
# 가중치가 이미 있으면 멈춘다. 덮어쓰면 그 회차를 같은 조건으로 다시 비교할 수 없다.
# 앞 실행이 중간에 끊겨 폴더만 남았으면 Drive 에서 그 폴더 이름을 바꿔(h1_failed1 등) 둔다.
if RUN_DIR.exists():
    raise RuntimeError(f"이미 결과가 있습니다(덮어쓰지 않음): {RUN_DIR}")
if REPORT.exists():  # 앞 실행이 결과를 남기기 전에 멈췄다 → 그 기록은 이름을 바꿔 둔다
    REPORT.rename(REPORT.with_name(f"{REPORT.stem}_failed_{int(time.time())}.txt"))

In [ ]:
# 3. 설치 · 압축 풀기 (1~2분)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}"], check=True
)
if not WORK.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(WORK)
if not (DATA / "data.yaml").exists():
    raise RuntimeError(f"zip 안에 yolo_{ROUND}/data.yaml 이 없습니다")

note(f"# JARVISEO 손 박스 검출기 {ROUND.upper()} 기록")
note(f"- 시각(UTC): {datetime.now(UTC):%Y-%m-%d %H:%M}")
note(f"- ultralytics {ULTRALYTICS} · GPU {GPU} · CPU {CPUS}개 · RAM {RAM_GB:.0f}GB")
note(f"- 학습: yolov8n.pt 에서 · 1종 hand · imgsz 640 · {EPOCHS} epoch · patience 0 · seed 0")
note()
note("## 학습 묶음 (맥에서 만든 기록)")
note((DATA / "prep_report.txt").read_text(encoding="utf-8"))

In [ ]:
# 4. 학습. 결과는 Drive hand_box/runs/h1 에 바로 저장된다(끊겨도 남음)
from ultralytics import YOLO

started = time.time()
# 검출기와 같은 구조(yolov8n)에서 시작한다
YOLO("yolov8n.pt").train(
    data=str(DATA / "data.yaml"),
    epochs=EPOCHS,
    # 0 = 조기 종료 없음. 끝까지 돌아야 학습률이 다 내려가고 마지막 10 epoch 의
    # 모자이크 끄기(close_mosaic)도 탄다. best.pt 는 그대로 검증 점수로 고른다.
    patience=0,
    imgsz=640,
    seed=0,
    deterministic=True,
    project=str(DRIVE / "runs"),
    name=ROUND,
)
train_minutes = (time.time() - started) / 60

In [ ]:
# 5. best.pt 를 검증 묶음(손 영상 loby · 검출기 d05)으로 다시 재고 보고서 마무리
with (RUN_DIR / "results.csv").open(encoding="utf-8") as f:
    rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
# best.pt 를 고르는 기준(fitness)과 같은 mAP50-95 로 best epoch 를 찾는다(검출기 노트북과 같음)
best = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
metrics = YOLO(str(RUN_DIR / "weights" / "best.pt")).val(
    data=str(DATA / "data.yaml"), imgsz=640, plots=False, project=str(WORK / "val"), name=ROUND
)
box = metrics.box
note()
note(f"## {ROUND} (best.pt · 검증 = 손 영상 loby · 검출기 d05)")
note(f"- 돈 epoch {len(rows)} / {EPOCHS} · best epoch {int(float(best['epoch']))}")
note(f"- 걸린 시간 {train_minutes:.1f}분")
note(f"- P {box.mp:.3f} · R {box.mr:.3f} · mAP50 {box.map50:.3f} · mAP50-95 {box.map:.3f}")
# 받을 파일 이름을 하나로 — best.pt 는 회차 · 모델마다 이름이 같아 다운로드 폴더에서 섞인다
WEIGHTS = DRIVE / f"hand_box_{ROUND}.pt"
shutil.copy2(RUN_DIR / "weights" / "best.pt", WEIGHTS)
note(f"가중치: {RUN_DIR / 'weights' / 'best.pt'} → 받을 사본 {WEIGHTS}")
note()
note("손 찾기 · 가리킴 평가(valid 102 · s04)는 맥에서 run_ablation.py --hand-boxes 로 잰다.")
print(f"\n끝. Drive JARVISEO/hand_box/ 의 {REPORT.name} · {WEIGHTS.name} 두 파일을 받는다.")

## 끝나면

- 맨 아래에 `끝.` 이 찍혔으면 성공이다. 결과 파일 두 개(둘 다 `JARVISEO/hand_box/` 바로 안)는 Aside 가 내려받는다.
  1. `HAND_BOX_H1_report.txt` — 이 실행의 기록
  2. `hand_box_h1.pt` — `runs/h1/weights/best.pt` 의 사본(약 6MB). 이름이 `best.pt` 면 다른 회차와 섞여서 사본을 받는다
- 중간에 빨간 오류로 멈췄으면 고치지 말고, 오류가 난 셀 번호와 마지막 20줄을 그대로 보낸다.